(sec-8)=
# 8. Conclusiones y limitaciones

In [31]:
ganancia_mejor = bt_ap.set_index("modelo").loc[MEJOR, "diferencia"]
equivalentes = [CORTO[k] for k in COMP[1:] if CORTO[k] not in sig_te]
adic_cols = [c for c in med.columns if c != "bank marketing"]
n_ad_gana = int(sum(r_l2e[c] == 1 for c in adic_cols))
display(Markdown(f"""## 8. Conclusiones y limitaciones

### 8.1 Hallazgos

1. **El límite de los 140 modelos del tercer entregable era temporal, no de flexibilidad.** La tasa diaria de suscripción es un nivel latente con memoria larga (autocorrelación de {num(fac[1], 2)} al primer rezago en el entrenamiento), y un puntaje que solo conoce la tasa de los días cerrados alcanza en el bucle externo un AUC-PR de {num(solo_nivel.iloc[:3]['Media'].max(), 3)}, frente a {num(solo_nivel.iloc[-1]['Media'], 3)} de la mejor de las 140 combinaciones. Ningún modelo estático puede representar esa estructura, y por eso los finalistas eran estadísticamente equivalentes.
2. **El L2E convierte ese hallazgo en un modelo.** Separa un nivel diario que se mueve rápido de unas pendientes que derivan despacio, se ajusta con un Newton por bloques de costo lineal en el número de días y se actualiza con un filtro de Laplace que respeta la demora de las etiquetas. La recalibración mensual que sugirió el docente es un caso límite de su filtro (propiedad P5).
3. **Desempeño.** En el bucle externo, el L2E obtiene {ms(ext['L2E'])}, frente a {ms(ext[MEJOR])} de la mejor combinación; en la prueba, un AUC-PR de {num(ap_te['L2E'], 3)} y un AUC de {num(auc_te['L2E'], 3)}, con una ganancia de {num(ganancia_mejor, 3)} en AUC-PR sobre la mejor combinación y probabilidades calibradas sin recalibración aparte (Brier de {num(resumen.iloc[0]['Brier prueba'], 3)}).
4. **Dónde está la ventaja.** Dentro de cada mes de prueba, el L2E es significativamente mejor que {enumerar(sig_te)}, y en el ordenamiento global de la prueba supera a todos los benchmarks. Frente a {enumerar(equivalentes)} no es superior dentro de cada mes: la ventaja viene de seguir el nivel de conversión entre días, no de ordenar mejor a los clientes de un mismo día.
5. **El mecanismo es transferible.** El filtro de nivel aislado mejora a {c2_gana} de los {len(COMP) - 1} benchmarks cuando se les aplica como envoltorio y supera a la recalibración mensual en {len(c2_vs_mensual)} de {len(con_mensual)} modelos con probabilidades; el mejor de esos híbridos queda a {num(bt_c['diferencia'], 3)} del L2E en AUC-PR de prueba ({p_txt(bt_c['p'])}).
6. **Límites de validez.** La ventaja se reduce con la demora de las etiquetas, y {'en ninguno de los tres conjuntos adicionales el L2E es el mejor modelo' if n_ad_gana == 0 else f'el L2E es el mejor modelo en {n_ad_gana} de los tres conjuntos adicionales'}: donde el nivel latente es débil o lo explican las variables, no supera a los benchmarks. El modelo es una herramienta para problemas de propensión con un nivel persistente, como el telemercadeo bancario, no un clasificador general.
7. **Mejoras evaluadas.** De los cinco ajustes para los árboles y las cinco mejoras del L2E (sección 6.13), {'ninguna supera la regla de decisión declarada, así que el L2E se conserva sin cambios' if (t_mej['Decisión'] == 'Se adopta').sum() == 0 else 'se adopta ' + enumerar(t_mej.loc[t_mej['Decisión'] == 'Se adopta', 'Mejora'].tolist())}; los ajustes que más ayudan a los modelos de referencia son los que usan las etiquetas recientes, el mecanismo del modelo original.

### 8.2 Veredicto sobre R1, R2 y R3

R1 (novedad en la aplicación) y R2 (implementación algorítmica propia) se cumplen. R3 {'se cumple' if r3 else 'no se cumple'} con la definición operativa de la guía: la media externa del L2E supera a la de los siete finalistas, {n_r3} pruebas distintas rechazan la igualdad frente a la mejor combinación benchmark con p ajustado < 0,05 y el efecto es {st.magnitud_cliff(cliff_mejor)} (δ de Cliff = {num(cliff_mejor, 3)}). Por la regla de la guía (sección 12), el tope que corresponde es {'5,0' if r3 else '4,0'}, sujeto a la evaluación del docente.

### 8.3 Limitaciones

- **La significancia depende de la unidad de análisis.** Con los cinco pliegues externos o con los meses de la validación externa ninguna prueba es significativa, porque en 2008 ningún modelo discrimina; la evidencia de superioridad viene de la prueba (2009-2010), que es el régimen de uso pero también un solo periodo.
- **Sin ventaja dentro de cada periodo frente a las mejores logísticas.** El L2E no ordena mejor a los clientes de un mismo mes que una logística bien regularizada o la SVM, y la prueba bayesiana se inclina levemente a favor de ellas; su aporte es el nivel.
- **Condición operativa.** El modelo necesita que las etiquetas de cada día estén disponibles poco después; con demoras de semanas pierde parte de su ventaja.
- **Aproximaciones.** Las pendientes se toman constantes dentro de la ventana de entrenamiento, la posterior se aproxima por Laplace y la probabilidad predictiva por la aproximación probit; ninguna se evaluó frente a una inferencia exacta (por ejemplo, MCMC).
- **Estabilidad de los coeficientes entre pliegues.** El L2E es menos estable que la logística con C = 0,01, porque la validación interna elige un λ distinto en cada pliegue (sección 1.2).
- **Selección de hiperparámetros con pocos datos.** En el primer pliegue de *electricity* la validación interna eligió una dinámica que no existía; con ventanas de ajuste pequeñas, el modelo puede perseguir ruido.
- **Revisión de literatura.** Se hizo con APIs públicas; Scopus y Web of Science no se consultaron por falta de acceso programático, y las búsquedas por relevancia no permiten intersecciones booleanas.
- **Uso de la prueba durante el prototipado.** Durante el desarrollo se evaluaron algunas variantes exploratorias también en la prueba. Las decisiones finales están respaldadas por la evidencia del entrenamiento y del bucle externo (secciones 2.2, 5.8 y 5.9), pero esa consulta previa es una forma de filtración de la prueba hacia el diseño y se declara aquí.

### 8.4 Trabajo futuro

- Validar con datos posteriores a 2010 el ensamble del L2E con un Random Forest reentrenado cada mes (mejora M4), que sube el AUC-PR de la prueba pero no el del bucle externo, y por eso no se adoptó (sección 6.13).
- Buscar una forma de elegir $q_u$ que funcione con ventanas de ajuste cortas: el Bayes empírico (mejora M2) aprende la dinámica del pasado y falla cuando el régimen cambia más de lo que cambiaba antes.
- Extender el nivel a varios grupos (por canal de contacto o por campaña) con un modelo jerárquico, y combinar el filtro con un clasificador no lineal como término fijo, ya que el componente C2 es transferible.
- Evaluar el modelo en datos posteriores a 2010 o en otros problemas de propensión con etiquetas diferidas (abandono de clientes, cobranza, conversión publicitaria).

### 8.5 Declaración de autoría y uso de herramientas de inteligencia artificial

Este trabajo es de autoría de Donny Torres. En su desarrollo se usó Claude (Anthropic) como asistente de programación, análisis y redacción: apoyo en la evaluación crítica del tercer entregable, en la formulación del modelo L2E y la derivación de su algoritmo, en la implementación del código, de las pruebas unitarias y de los *scripts* de experimentación, en la búsqueda y verificación bibliográfica, en la integración del modelo al dashboard, en el armado de este Jupyter Book y en la redacción de los textos. Todas las cifras se calculan con el código incluido y se reproducen con los comandos de la sección 9. El autor revisó el código, las derivaciones y los resultados, y es responsable de comprender, explicar y modificar cualquier parte del algoritmo y de las pruebas estadísticas, como exige la guía (secciones 5.4 y 13).

**Firma:** Donny Torres · Maestría en Analítica de Datos, Universidad del Norte · Barranquilla, 24 de octubre de 2026."""))

(sec-8-1)=
## 8.1 Hallazgos

1. **El límite de los 140 modelos del tercer entregable era temporal, no de flexibilidad.** La tasa diaria de suscripción es un nivel latente con memoria larga (autocorrelación de 0,76 al primer rezago en el entrenamiento), y un puntaje que solo conoce la tasa de los días cerrados alcanza en el bucle externo un AUC-PR de 0,103, frente a 0,103 de la mejor de las 140 combinaciones. Ningún modelo estático puede representar esa estructura, y por eso los finalistas eran estadísticamente equivalentes.
2. **El L2E convierte ese hallazgo en un modelo.** Separa un nivel diario que se mueve rápido de unas pendientes que derivan despacio, se ajusta con un Newton por bloques de costo lineal en el número de días y se actualiza con un filtro de Laplace que respeta la demora de las etiquetas. La recalibración mensual que sugirió el docente es un caso límite de su filtro (propiedad P5).
3. **Desempeño.** En el bucle externo, el L2E obtiene 0,111 ± 0,116, frente a 0,103 ± 0,108 de la mejor combinación; en la prueba, un AUC-PR de 0,602 y un AUC de 0,772, con una ganancia de 0,068 en AUC-PR sobre la mejor combinación y probabilidades calibradas sin recalibración aparte (Brier de 0,171).
4. **Dónde está la ventaja.** Dentro de cada mes de prueba, el L2E es significativamente mejor que Naive Bayes, XGBoost y Árbol de decisión, y en el ordenamiento global de la prueba supera a todos los benchmarks. Frente a Logística L1/L2, Random Forest, SVM, KNN y Referencia E2 no es superior dentro de cada mes: la ventaja viene de seguir el nivel de conversión entre días, no de ordenar mejor a los clientes de un mismo día.
5. **El mecanismo es transferible.** El filtro de nivel aislado mejora a 6 de los 8 benchmarks cuando se les aplica como envoltorio y supera a la recalibración mensual en 5 de 7 modelos con probabilidades; el mejor de esos híbridos queda a 0,008 del L2E en AUC-PR de prueba (p = 0,070).
6. **Límites de validez.** La ventaja se reduce con la demora de las etiquetas, y en ninguno de los tres conjuntos adicionales el L2E es el mejor modelo: donde el nivel latente es débil o lo explican las variables, no supera a los benchmarks. El modelo es una herramienta para problemas de propensión con un nivel persistente, como el telemercadeo bancario, no un clasificador general.
7. **Mejoras evaluadas.** De los cinco ajustes para los árboles y las cinco mejoras del L2E (sección {ref}`6.13 <sec-6-13>`), ninguna supera la regla de decisión declarada, así que el L2E se conserva sin cambios; los ajustes que más ayudan a los modelos de referencia son los que usan las etiquetas recientes, el mecanismo del modelo original.

(sec-8-2)=
## 8.2 Veredicto sobre R1, R2 y R3

R1 (novedad en la aplicación) y R2 (implementación algorítmica propia) se cumplen. R3 se cumple con la definición operativa de la guía: la media externa del L2E supera a la de los siete finalistas, 3 pruebas distintas rechazan la igualdad frente a la mejor combinación benchmark con p ajustado < 0,05 y el efecto es pequeño (δ de Cliff = 0,156). Por la regla de la guía (sección 12), el tope que corresponde es 5,0, sujeto a la evaluación del docente.

(sec-8-3)=
## 8.3 Limitaciones

- **La significancia depende de la unidad de análisis.** Con los cinco pliegues externos o con los meses de la validación externa ninguna prueba es significativa, porque en 2008 ningún modelo discrimina; la evidencia de superioridad viene de la prueba (2009-2010), que es el régimen de uso pero también un solo periodo.
- **Sin ventaja dentro de cada periodo frente a las mejores logísticas.** El L2E no ordena mejor a los clientes de un mismo mes que una logística bien regularizada o la SVM, y la prueba bayesiana se inclina levemente a favor de ellas; su aporte es el nivel.
- **Condición operativa.** El modelo necesita que las etiquetas de cada día estén disponibles poco después; con demoras de semanas pierde parte de su ventaja.
- **Aproximaciones.** Las pendientes se toman constantes dentro de la ventana de entrenamiento, la posterior se aproxima por Laplace y la probabilidad predictiva por la aproximación probit; ninguna se evaluó frente a una inferencia exacta (por ejemplo, MCMC).
- **Estabilidad de los coeficientes entre pliegues.** El L2E es menos estable que la logística con C = 0,01, porque la validación interna elige un λ distinto en cada pliegue (sección {ref}`1.2 <sec-1-2>`).
- **Selección de hiperparámetros con pocos datos.** En el primer pliegue de *electricity* la validación interna eligió una dinámica que no existía; con ventanas de ajuste pequeñas, el modelo puede perseguir ruido.
- **Revisión de literatura.** Se hizo con APIs públicas; Scopus y Web of Science no se consultaron por falta de acceso programático, y las búsquedas por relevancia no permiten intersecciones booleanas.
- **Uso de la prueba durante el prototipado.** Durante el desarrollo se evaluaron algunas variantes exploratorias también en la prueba. Las decisiones finales están respaldadas por la evidencia del entrenamiento y del bucle externo (secciones {ref}`2.2 <sec-2-2>`, {ref}`5.8 <sec-5-8>` y {ref}`5.9 <sec-5-9>`), pero esa consulta previa es una forma de filtración de la prueba hacia el diseño y se declara aquí.

(sec-8-4)=
## 8.4 Trabajo futuro

- Validar con datos posteriores a 2010 el ensamble del L2E con un Random Forest reentrenado cada mes (mejora M4), que sube el AUC-PR de la prueba pero no el del bucle externo, y por eso no se adoptó (sección {ref}`6.13 <sec-6-13>`).
- Buscar una forma de elegir $q_u$ que funcione con ventanas de ajuste cortas: el Bayes empírico (mejora M2) aprende la dinámica del pasado y falla cuando el régimen cambia más de lo que cambiaba antes.
- Extender el nivel a varios grupos (por canal de contacto o por campaña) con un modelo jerárquico, y combinar el filtro con un clasificador no lineal como término fijo, ya que el componente C2 es transferible.
- Evaluar el modelo en datos posteriores a 2010 o en otros problemas de propensión con etiquetas diferidas (abandono de clientes, cobranza, conversión publicitaria).

(sec-8-5)=
## 8.5 Declaración de autoría y uso de herramientas de inteligencia artificial

Este trabajo es de autoría de Donny Torres. En su desarrollo se usó Claude (Anthropic) como asistente de programación, análisis y redacción: apoyo en la evaluación crítica del tercer entregable, en la formulación del modelo L2E y la derivación de su algoritmo, en la implementación del código, de las pruebas unitarias y de los *scripts* de experimentación, en la búsqueda y verificación bibliográfica, en la integración del modelo al dashboard, en el armado de este Jupyter Book y en la redacción de los textos. Todas las cifras se calculan con el código incluido y se reproducen con los comandos de la sección 9. El autor revisó el código, las derivaciones y los resultados, y es responsable de comprender, explicar y modificar cualquier parte del algoritmo y de las pruebas estadísticas, como exige la guía (secciones {ref}`5.4 <sec-5-4>` y 13).

**Firma:** Donny Torres · Maestría en Analítica de Datos, Universidad del Norte · Barranquilla, 24 de octubre de 2026.